# 逐步运行临床分析（拆分版）

本 notebook 用于把 `run_full_clinical_analysis()` 的每一步拆开单独运行，便于检查：

- 数据是否合并正确（SubjectID/Cluster/协变量）
- 每个模块函数是否工作正常
- 输出结果（表格/图）是否符合预期

你只需要在“路径设置”单元格里改 `cluster_file` 和 `clinical_dir` 即可切换不同的聚类结果。

In [ ]:
# 路径与导入
import sys
from pathlib import Path

nb_dir = Path.cwd()          # .../src/notebook
src_dir = nb_dir.parent      # .../src
proj_dir = src_dir.parent    # 项目根目录

if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

from clinical_analysis import (
    get_config,
    build_analysis_dataset,
)
from clinical_analysis import (
    run_rf_feature_selection,
    run_kde_visualization,
    run_ancova_models,
    run_ks_distribution_tests,
    run_quantile_models,
    run_logistic_models,
)

print("proj_dir:", proj_dir)
print("src_dir:", src_dir)

In [ ]:
# 需要你改的路径（切换不同 cluster 结果主要改这里）

data_dir = proj_dir / "data"

# cluster_file: 至少包含 SubjectID 和 Cluster
cluster_file = data_dir / "cluster" / "average2.csv"  # 按你当前结构

# clinical_dir: 包含 clinical_final 的那些固定临床数据文件
clinical_dir = data_dir / "clinical_final"  

# 输出目录（每个 cluster 一个子目录）
output_dir = data_dir / "results" / Path(cluster_file).stem

print("cluster_file:", cluster_file)
print("clinical_dir:", clinical_dir)
print("output_dir:", output_dir)

In [ ]:
# 配置（可在这里改 quantiles / 是否使用时间差协变量等）
cfg = get_config(
    quantiles=[0.25, 0.50, 0.75],
    time_diff_in_models=True,
)


In [ ]:
# Step 1) 构建统一数据集（合并 cluster + 临床数据）
datasets = build_analysis_dataset(
    cluster_path=cluster_file,
    clinical_dir=clinical_dir,
    config=cfg,
)

for k in ["cluster_info", "biochem", "cbc", "ecg", "diagnosis"]:
    df = datasets.get(k)
    if df is None:
        print(f"{k}: None")
    else:
        print(f"{k}: shape={df.shape} | cols={len(df.columns)}")

# 快速检查 cluster 分布
ci = datasets["cluster_info"]
print("\nCluster counts:")
print(ci[cfg["cluster_label_col"]].value_counts().sort_index())

ci.head()

In [ ]:
# Step 2) 特征筛选（RF 排列重要性）+ 线性(LR) vs 非线性(RF) 对比
rf_out = run_rf_feature_selection(datasets, config=cfg, cv_folds=5, random_state=42, n_repeats=20)

import pandas as pd

rf_summary_df = pd.DataFrame(rf_out["summary"]).sort_values("category") if rf_out["summary"] else pd.DataFrame()
rf_summary_df

In [ ]:
# 查看每一类数据的 Top 10 重要特征
for cat, imp_df in rf_out["importance"].items():
    print("\n===", cat, "===")
    display(imp_df.head(10))

In [ ]:
# Step 3) 分布探索：KDE 曲线（默认每组取前 6 个变量画图）
from pathlib import Path

fig_dir = Path(output_dir) / "figures"
fig_dir.mkdir(parents=True, exist_ok=True)

_ = run_kde_visualization(
    datasets,
    config=cfg,
    output_dir=fig_dir,
    top_n_per_group=6,
)

print("KDE figures saved to:", fig_dir)

In [ ]:
# Step 4) 均值效应检验：ANCOVA / OLS
ancova_df = run_ancova_models(datasets, config=cfg, fdr_alpha=0.05)
ancova_df.sort_values(["data", "p_fdr"]).head(20)

In [ ]:
# Step 5) 分布差异检验：KS（两两 cluster 比较）
ks_df = run_ks_distribution_tests(datasets, config=cfg, fdr_alpha=0.05)
ks_df.sort_values(["data", "p_fdr"]).head(30)

In [ ]:
# Step 6) 分位数差异：Quantile regression (0.25/0.50/0.75)
q_df = run_quantile_models(datasets, config=cfg, quantiles=cfg["quantiles"], fdr_alpha=0.05)
q_df.sort_values(["data", "quantile", "p_fdr"]).head(30)

In [ ]:
# Step 7) 二分类结局：ECG / 诊断 Logistic（若对应数据存在）
logit_df = run_logistic_models(datasets, config=cfg, fdr_alpha=0.05)
logit_df.sort_values(["data", "p_fdr"]).head(30)

In [ ]:
tables_dir = Path(output_dir) / "tables"
tables_dir.mkdir(parents=True, exist_ok=True)
logit_df.to_csv(tables_dir / "logistic_results.csv", index=False)

In [ ]:
# Step 8) 导出表格（可选）
import pandas as pd

tables_dir = Path(output_dir) / "tables"
tables_dir.mkdir(parents=True, exist_ok=True)

if not rf_summary_df.empty:
    rf_summary_df.to_csv(tables_dir / "rf_cv_summary.csv", index=False)
for cat, imp_df in rf_out["importance"].items():
    imp_df.to_csv(tables_dir / f"rf_importance_{cat}.csv", index=False)

if ancova_df is not None and not ancova_df.empty:
    ancova_df.to_csv(tables_dir / "ancova_results.csv", index=False)
if ks_df is not None and not ks_df.empty:
    ks_df.to_csv(tables_dir / "ks_distribution_tests.csv", index=False)
if q_df is not None and not q_df.empty:
    q_df.to_csv(tables_dir / "quantile_regression_results.csv", index=False)
if logit_df is not None and not logit_df.empty:
    logit_df.to_csv(tables_dir / "logistic_results.csv", index=False)

print("Tables saved to:", tables_dir)
print("Figures saved to:", fig_dir)